# 🏫 University GPU Training (Constrained Environment)
    
**Hardware Allocation Limits:**
*   **CPU:** 1 Core
*   **RAM:** 8 GB
*   **GPU:** 16% Fractional Access (~3.8GB on RTX6000, ~12.8GB on A100)
*   **Time:** 4 Hours Session

This notebook strictly isolates PyTorch to your exact constraints so you avoid Out-Of-Memory (OOM) crashing or session banning.

In [ ]:
import os
import torch

# 1. Enforce CPU Core Limits (Prevent Thread-bombing the cluster)
torch.set_num_threads(1)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# 2. Enforce GPU Memory Allocation constraints (16% fraction)
if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.16)
    print("✅ PyTorch GPU Memory locked to 16% fraction.")
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected? Ensure you spawned a GPU Jupyter session.")
    
print("✅ CPU Threads locked to 1 processing core.")

### Execute Training Runner
We execute the model pointing to our specially designed `nyc_lpe_stgtn_university_gpu.yaml`. This YAML halves the batch size (`16`) so we comfortably fit inside the 8GB RAM restriction without paging.

In [ ]:
from pathlib import Path
from lpe_stgtn.training.lpe_stgtn_runner import run_lpe_stgtn_experiment

# Define paths
project_root = Path.cwd()
experiment_config_path = project_root / "configs" / "experiments" / "nyc_lpe_stgtn_university_gpu.yaml"
training_config_path = project_root / "configs" / "training" / "default.yaml"

# Set up fallback empty default training config if missing
if not training_config_path.exists():
    training_config_path.parent.mkdir(parents=True, exist_ok=True)
    training_config_path.write_text("{}")

# Run the localized Model!
print("🚀 Launching University-Constrained Run...")
summary = run_lpe_stgtn_experiment(
    experiment_config_path=experiment_config_path,
    project_root=project_root,
    training_config_path=training_config_path
)

print("\n🎉 Run Completed!")
print(f"Experiment: {summary.experiment_name}")
print(f"Test MAE: {summary.metrics['test']['mae']:.4f}")
print(f"Test RMSE: {summary.metrics['test']['rmse']:.4f}")
print("Checkpoints saved safely.")
